<a href="https://colab.research.google.com/github/Sprg72/Data-Engineer/blob/main/notebooks/Pyspark_for_datafabric1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# python, pandas, numpy  --> already available with Google colab notebooks




```
#step1
!pip install findspark pyspark

#step2
import findspark
findspark.init()

#step3
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('myapp').getOrCreate()

```



In [2]:
# step1
!pip install findspark pyspark

In [3]:
# step2
import findspark
findspark.init()

In [4]:
# step3
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('myapp').getOrCreate()

In [5]:
spark

In [6]:
# spark data objects are called 'RDD's
# on RDD two operations --> 1. transformations  2. actions
# 3 ways to create RDD's
# Way 1: when you parallelize local object using sparkCOntext (sc), a new RDD will be created.
# way 2: when you read from a file using sparkContext (sc), a new RDD will be created.
# way 3: when you perform some transformations on existing RDD, a new RDD will be created.

In [7]:
# way1 : when you parallelize local object using sparkCOntext (sc), a new RDD will be created.
x = [10,20,30,40,50,60,70,80,90,100]
len(x)

10

In [8]:
type(x)  # local object (list)

list

In [9]:
sc = spark.sparkContext
sc

<SparkContext master=local[*] appName=myapp>

In [10]:
R1 = sc.parallelize(x)
type(R1)

pyspark.core.rdd.RDD

In [11]:
R1.getNumPartitions()

2

In [12]:
R1.glom().collect()

[[10, 20, 30, 40, 50], [60, 70, 80, 90, 100]]

In [13]:
R1.repartition(3)

MapPartitionsRDD[6] at coalesce at NativeMethodAccessorImpl.java:0

In [14]:
R1.getNumPartitions()

2

In [15]:
R1=R1.repartition(3)
R1.getNumPartitions()

3

In [16]:
R1 = R1.coalesce(1)
R1.getNumPartitions()

1

In [17]:
# RDD.repartition() # we optimized for increasing no. of partitions
# RDD.coalesec() # optimized for decreasing no. of partitions

In [18]:
# way2: when you read from a file using sparkContext (sc), a new RDD will be created.



```
# input file : emp1.txt

101, amar,90000,m,11
102, amala,20000,f,12
103, ankit,40000,m,13
104, ankita,60000,f,13
105, anusha,110000,f,12
106, anuz,20000,m,11
107, akash,100000,m,12
108, siva,20000,m,14
109, sivani,30000,f,15
110, mani,30000,m,12
111, manisha,300000,f,13
112, sivam,200000,m,12
113, varun,200000,m,13
```



In [23]:
data = sc.textFile('/content/emp1.txt')
type(data)

pyspark.core.rdd.RDD

In [24]:
data.collect() # action

['101, amar,90000,m,11',
 '102, amala,20000,f,12',
 '103, ankit,40000,m,13',
 '104, ankita,60000,f,13',
 '105, anusha,110000,f,12',
 '106, anuz,20000,m,11',
 '107, akash,100000,m,12',
 '108, siva,20000,m,14',
 '109, sivani,30000,f,15',
 '110, mani,30000,m,12',
 '111, manisha,300000,f,13',
 '112, sivam,200000,m,12',
 '113, varun,200000,m,13']

In [25]:
# way 3: when you perform some transformations on existing RDD, a new RDD will be created.
R1.collect()

[10, 20, 30, 40, 50, 60, 70, 80, 90, 100]

In [26]:
R2 = R1.map(lambda x : x + 100)
type(R2)

pyspark.core.rdd.PipelinedRDD

In [27]:
R2.collect()
# DAG flow : R1 --> R2

[110, 120, 130, 140, 150, 160, 170, 180, 190, 200]

In [28]:
R3 = R2.filter(lambda x: x > 150)
type(R3)

pyspark.core.rdd.PipelinedRDD

In [29]:
R3.collect()
#DAG flow : R1 --> R2 --> R3


[160, 170, 180, 190, 200]

In [30]:
# a demo on structured data aggregations.
# from emp1.txt
# task 1:  find total salary of the organisation
# sql : select sum(salary) from emp;

In [31]:
emp = sc.textFile('/content/emp1.txt')
emp.collect()

['101, amar,90000,m,11',
 '102, amala,20000,f,12',
 '103, ankit,40000,m,13',
 '104, ankita,60000,f,13',
 '105, anusha,110000,f,12',
 '106, anuz,20000,m,11',
 '107, akash,100000,m,12',
 '108, siva,20000,m,14',
 '109, sivani,30000,f,15',
 '110, mani,30000,m,12',
 '111, manisha,300000,f,13',
 '112, sivam,200000,m,12',
 '113, varun,200000,m,13']

In [32]:
sal = emp.map(lambda x : int(x.split(',')[2]))  # transformation

sal.collect()
tot = sal.sum() # action
print("Total salary ", tot)

Total salary  1220000


In [33]:
#task 2: for each gender, find number of employees.
#sql : select gender, count(*) from emp group by gender
emp.collect()

['101, amar,90000,m,11',
 '102, amala,20000,f,12',
 '103, ankit,40000,m,13',
 '104, ankita,60000,f,13',
 '105, anusha,110000,f,12',
 '106, anuz,20000,m,11',
 '107, akash,100000,m,12',
 '108, siva,20000,m,14',
 '109, sivani,30000,f,15',
 '110, mani,30000,m,12',
 '111, manisha,300000,f,13',
 '112, sivam,200000,m,12',
 '113, varun,200000,m,13']

In [34]:
# to perform grouping aggregations, RDD.reduceByKey() transformation is used.
# NOte: for reduceByKey() inout should be pairRDD.
# what is pairRDD ---> is a tuple of key and value. eg: pair --> [("m", 30000),("f",40000), ("m, 50000), ....]
# in the tuple, first one is key, 2nd one is value
# eg: task: for gender find total salary
# for the given task, group by 'gender' so gender should be key
# aggregation function on 'salary' ---> so salary should be value

In [35]:
words = emp.map(lambda x : x.split(','))
words.collect()

[['101', ' amar', '90000', 'm', '11'],
 ['102', ' amala', '20000', 'f', '12'],
 ['103', ' ankit', '40000', 'm', '13'],
 ['104', ' ankita', '60000', 'f', '13'],
 ['105', ' anusha', '110000', 'f', '12'],
 ['106', ' anuz', '20000', 'm', '11'],
 ['107', ' akash', '100000', 'm', '12'],
 ['108', ' siva', '20000', 'm', '14'],
 ['109', ' sivani', '30000', 'f', '15'],
 ['110', ' mani', '30000', 'm', '12'],
 ['111', ' manisha', '300000', 'f', '13'],
 ['112', ' sivam', '200000', 'm', '12'],
 ['113', ' varun', '200000', 'm', '13']]

In [36]:
pair = words.map(lambda x : (x[-2],1))
pair.collect()

[('m', 1),
 ('f', 1),
 ('m', 1),
 ('f', 1),
 ('f', 1),
 ('m', 1),
 ('m', 1),
 ('m', 1),
 ('f', 1),
 ('m', 1),
 ('f', 1),
 ('m', 1),
 ('m', 1)]

In [37]:
gendcnt = pair.reduceByKey(lambda x,y : x+y)

In [43]:
gendcnt.collect()

[('m', 8), ('f', 5)]



```
# how reduceByKey works:
gendcnt = pair.reduceByKey(lambda x,y : x+y)

# groups the data
(m, [1,1,1,1,1,1,1,1])
# cumulative sum
iteration 1:
x=1, y=1  ---> x+y=2
iteration 2:
x=2, y= 1 ---> x+y=3
iteration 3:
x=3, y= 1 ---> x+y=4
iteration 4:
x=4, y= 1 ---> x+y=5
iteration 5:
x=5, y= 1 ---> x+y=6
iteration 6:
x=6, y= 1 ---> x+y=7
iteration 7:
x=7, y= 1 ---> x+y=8

(f, [1, 1, 1, 1, 1])
# same thing will happend for female.
(f,5)


emp = sc.textFile('/content/emp1.txt')
words = emp.map(lambda x : x.split(','))
pair = words.map(lambda x : (x[-2],1))
gendcnt = pair.reduceByKey(lambda x,y : x+y)
gendcnt.collect()

```




In [39]:
emp = sc.textFile('/content/emp1.txt')
words = emp.map(lambda x : x.split(','))
pair = words.map(lambda x : (x[-2],1))
gendcnt = pair.reduceByKey(lambda x,y : x+y)
gendcnt.collect()

[('m', 8), ('f', 5)]

In [41]:
# task 3: for each gender find total salary
# sql: select gender, sum(salary) from emp group by gender;

gend_sal_pair = words.map(lambda x : (x[-2], int(x[2])))
gend_sal_pair.collect()


[('m', 90000),
 ('f', 20000),
 ('m', 40000),
 ('f', 60000),
 ('f', 110000),
 ('m', 20000),
 ('m', 100000),
 ('m', 20000),
 ('f', 30000),
 ('m', 30000),
 ('f', 300000),
 ('m', 200000),
 ('m', 200000)]

In [42]:
gendtot = gend_sal_pair.reduceByKey(lambda x,y : x+y)
gendtot.collect()

[('m', 700000), ('f', 520000)]



```
# how reduceByKey works for sum aggregation
[('m', 90000),
 ('f', 20000),
 ('m', 40000),
 ('f', 60000),
 ('f', 110000),
 ('m', 20000),
 ('m', 100000),
 ('m', 20000),
 ('f', 30000),
 ('m', 30000),
 ('f', 300000),
 ('m', 200000),
 ('m', 200000)]

 gendtot = gend_sal_pair.reduceByKey(lambda x,y : x+y)

 ('f', [20000, 60000, 110000, 300000, 300000])

 iteration 1:
 x=20000, y=60000   ---> x+y= 80000
 iteration 2:
 x=80000, y=110000  ---> x+y= 190000
 iteration 3:
 x=190000, y=300000 ---> x+y= 490000
 iteration 4:
 x=490000, y=300000 ---> x+y= 790000 ---> its final result for 'female'

 -----> same thing done for 'male'

```

